# Stage 2 — Feature Table Extraction (Validation)

**External step**: Use MICS software to extract per-cell feature tables from raw TIFFs + labeled TIFFs (Stage 1 output).

**This notebook validates** that the expected CSV files exist, have the correct schema, and are ready for Stage 3.

In [1]:
import os
import pandas as pd
from pathlib import Path

from pipeline.config import load_config
from pipeline.io import discover_feature_files

cfg = load_config("config.yaml")
print(f"Experiment: {cfg.experiment_name}")

Experiment: my_experiment


## Validate Feature CSVs

For each machine, check that:
1. The `feature_csv_dir` exists and contains CSV files
2. CSVs have the expected marker columns
3. No critical data issues (all-NaN columns, missing cell IDs)

In [2]:
all_ok = True

for machine_name, machine_cfg in cfg.machines.items():
    csv_dir = machine_cfg.feature_csv_dir
    print(f"\n{'='*60}")
    print(f"Machine: {machine_name}")
    print(f"Feature CSV dir: {csv_dir}")
    print(f"{'='*60}")

    if not csv_dir or not os.path.isdir(csv_dir):
        print(f"  ✗ Directory not found!")
        all_ok = False
        continue

    feature_files, region_list = discover_feature_files(csv_dir)
    print(f"  Found {len(feature_files)} CSV file(s)")

    if len(feature_files) == 0:
        print(f"  ✗ No CSV files found!")
        all_ok = False
        continue

    # Check first file for schema
    sample_path = os.path.join(csv_dir, feature_files[0])
    sample_df = pd.read_csv(sample_path, nrows=5)
    sample_df.columns = sample_df.columns.str.replace("Biomarker Exp", "", regex=False).str.strip()

    print(f"  Sample columns ({len(sample_df.columns)} total): {list(sample_df.columns[:5])}...")

    # Check required meta columns
    for meta_col in cfg.meta_columns[:3]:  # Cell Id, Nuc X, Nuc Y Inv
        if meta_col not in sample_df.columns:
            print(f"  ✗ Missing required column: {meta_col}")
            all_ok = False

    # Check selected markers
    missing_markers = [m for m in cfg.selected_markers if m not in sample_df.columns]
    if missing_markers:
        print(f"  ✗ Missing {len(missing_markers)} selected markers: {missing_markers[:3]}...")
        all_ok = False
    else:
        print(f"  ✓ All {len(cfg.selected_markers)} selected markers present")

    # Report per-file row counts
    for ff in feature_files:
        fp = os.path.join(csv_dir, ff)
        n_rows = sum(1 for _ in open(fp)) - 1  # subtract header
        print(f"    {ff}: {n_rows} cells")

print(f"\n{'='*60}")
if all_ok:
    print("✓ All validations passed – ready for Stage 3")
else:
    print("✗ Some validations failed – fix issues before proceeding")


Machine: machine_A
Feature CSV dir: ./data/raw/mrd
  Found 10 CSV file(s)
  Sample columns (93 total): ['Cell Id', 'Nuc X', 'Nuc Y Inv', 'Actin REAL650', 'AKT Pan REA676']...
  ✓ All 25 selected markers present
    ./10_SN141_slides2_ROI-16.csv: 6642 cells
    ./1_SN177_ROI-06.csv: 10463 cells
    ./2_SN177_ROI-07.csv: 10162 cells
    ./3_SN177_ROI-09.csv: 4900 cells
    ./4_SN177_ROI-13.csv: 1853 cells
    ./5_SN177_ROI-14.csv: 3186 cells
    ./6_SN177_ROI-16.csv: 4849 cells
    ./7_SN141_slides1_ROI-07.csv: 6836 cells
    ./8_SN141_slides2_ROI-03.csv: 7625 cells
    ./9_SN141_slides2_ROI-06.csv: 2586 cells

Machine: machine_B
Feature CSV dir: ./data/raw/mrd
  Found 10 CSV file(s)
  Sample columns (93 total): ['Cell Id', 'Nuc X', 'Nuc Y Inv', 'Actin REAL650', 'AKT Pan REA676']...
  ✓ All 25 selected markers present
    ./10_SN141_slides2_ROI-16.csv: 6642 cells
    ./1_SN177_ROI-06.csv: 10463 cells
    ./2_SN177_ROI-07.csv: 10162 cells
    ./3_SN177_ROI-09.csv: 4900 cells
    ./4_SN17